# 02 — Silver validation: master-data 6-gate quality contract

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 — T2.4 |
| **Layer** | `silver/master-data/` |
| **Source** | `bronze.<table>` (Delta, managed) |
| **Target** | `silver.<table>` (Delta, managed, validated) |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md), [ADR-0016 Gate 2 — Ingestion PHI regex](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md), [ADR-0013 dual-mode residency](../../../docs/adr/0013-temporary-us-region-demo-scope.md) |
| **Design spec** | [sprint-09 §2.2 Notebook 02](../../../docs/sprints/sprint-09-master-data-simulation-and-capacity-dashboard.md) and [design §3.2 validation gates](../../../docs/superpowers/specs/2026-06-29-sprint09-master-data-capacity-dashboard-design.md) |

## Purpose

Apply the 6-gate quality contract to every bronze master-data table. Rows that
violate a gate are quarantined (rejected) rather than propagated to silver. Any
table-level failure (e.g. row count zero, PHI hit) aborts the notebook.

## The 6 validation gates

| # | Gate | Contract | Fail action |
| - | ---- | -------- | ----------- |
| 1 | Row count | `count(*) > 0` per table | Abort |
| 2 | Schema | Table conforms to `dc-master-XX-v1` (PK columns present, non-null) | Abort |
| 3 | PHI regex sweep | No **source** cell matches email / phone / DOB / CH AHV-13 patterns (ADR-0016 Gate 2) | Reject rows + alert; abort on any hit |
| 4 | Residency | `_residency_tag ∈ {CH-North, US-West}` (RB-01 dual-mode) | Abort |
| 5 | Data quality | `_data_quality ∈ {explicit, inferred, missing}` | Abort |
| 6 | FK integrity | Every foreign key value exists in parent dim | Abort if > 5% orphan |

Gates 3–5 also **derive** the silver-layer governance columns (`_residency_tag`,
`_data_quality`) from source-table values where present, defaulting to demo-scope
safe values otherwise. Gate 3 scans only **source** columns — the leading-`_`
governance/lineage columns (`_lineage_ref`, added by bronze, contains an ISO
timestamp) are excluded so the DOB regex does not false-positive on the lineage
stamp. Gate 3 is deterministic (pure regex, no LLM); its test cell at the end of
this notebook seeds a synthetic bad row and asserts rejection.

In [ ]:
target_lakehouse = 'lh_ihzhhpf_sit'
run_id = 'run-manual-local'
orphan_fk_threshold = 0.05  # 5% per design spec §3.2
default_residency_tag = 'US-West'  # demo scope per ADR-0013 (westus2 carve-out until 2026-09-30)

In [ ]:
# Master-data table registry (matches bronze notebook).
TABLES = [
    ('01_dim_hospital.csv',                       'dim_hospital',                             ['hospital_id'],                       {},                                                                                      'residency_tag', 'beds_quality'),
    ('02_dim_specialty.csv',                      'dim_specialty',                            ['specialty_hospital_id'],             {'hospital_id': 'dim_hospital'},                                                         None,             'data_quality'),
    ('03_dim_hospital_service.csv',               'dim_hospital_service',                     ['service_id'],                        {'hospital_id': 'dim_hospital', 'specialty_id': 'dim_specialty'},                        None,             'data_quality'),
    ('04_dim_disease.csv',                        'dim_disease',                              ['disease_id'],                        {},                                                                                      None,             None),
    ('05_dim_treatment.csv',                      'dim_treatment',                            ['treatment_id'],                      {'disease_id': 'dim_disease'},                                                           None,             None),
    ('06_dim_drg.csv',                            'dim_drg',                                  ['drg_code'],                          {'disease_id': 'dim_disease'},                                                           None,             None),
    ('07_dim_ward_capacityunit.csv',              'dim_ward_capacityunit',                    ['ward_id'],                           {'hospital_id': 'dim_hospital', 'specialty_id': 'dim_specialty'},                        None,             'bed_count_quality'),
    ('08_fact_capacity_baseline.csv',             'fact_capacity_baseline',                   ['hospital_id', 'metric'],             {'hospital_id': 'dim_hospital'},                                                         None,             'data_quality'),
    ('09_map_disease_treatment_specialty_service.csv', 'map_disease_treatment_specialty_service', ['map_id'],                       {'hospital_id': 'dim_hospital', 'disease_id': 'dim_disease', 'treatment_id': 'dim_treatment', 'drg_code': 'dim_drg', 'specialty_id': 'dim_specialty'}, None, 'data_quality'),
]

In [ ]:
# PHI regex catalogue — ADR-0016 Gate 2. Patterns are deliberately conservative:
# the demo dataset is synthetic reference data (no persons), so any positive match is a bug.
import re

PHI_PATTERNS = {
    'email':      re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+'),
    'phone':      re.compile(r'\+?\d[\d\s().-]{6,}'),
    'dob':        re.compile(r'\d{4}-\d{2}-\d{2}'),
    'ch_ahv_13':  re.compile(r'756\.\d{4}\.\d{4}\.\d{2}'),
}

# Structural envelope columns exempt from PHI regex scan.
# These are ISO timestamps / IDs shaped like PHI regex patterns but are
# always synthetic/system-generated. See ADR-0016 gate 2 rationale.
# Mirrors the allowlist in 02_silver_eventstream.ipynb for cross-notebook parity;
# master-data tables don't currently carry these columns, but the class of bug
# (regex tripping on synthetic timestamps/IDs) is identical and consistency matters.
STRUCTURAL_STRING_ALLOWLIST = {
    'simulatedAt',       # eventstream envelope timestamp
    'emittedAt',         # eventstream envelope timestamp
    'asOfTimestamp',     # DC-* contract standard field
    'eventId',           # UUID-shaped, may trip regex
    'simRunId',          # UUID-shaped
    # add more as new envelope shapes emerge
}

ALLOWED_RESIDENCY_TAGS = {'CH-North', 'US-West'}
ALLOWED_DATA_QUALITY = {'explicit', 'inferred', 'missing'}


In [ ]:
from pyspark.sql import DataFrame, functions as F, types as T
from datetime import datetime, timezone

class GateFailure(Exception):
    pass

def _now_iso() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def gate1_row_count(df: DataFrame, table: str) -> int:
    n = df.count()
    if n <= 0:
        raise GateFailure(f'{table}: gate 1 row_count failed (rows={n})')
    return n

def gate2_schema(df: DataFrame, table: str, pk_cols) -> None:
    missing = [c for c in pk_cols if c not in df.columns]
    if missing:
        raise GateFailure(f'{table}: gate 2 schema failed — missing PK cols {missing}')
    for c in pk_cols:
        n_null = df.filter(F.col(c).isNull()).count()
        if n_null > 0:
            raise GateFailure(f'{table}: gate 2 schema failed — {n_null} null value(s) in PK col {c}')

def gate3_phi_scan(df: DataFrame, table: str):
    """Scan every source string column against PHI_PATTERNS. Returns (clean_df, rejected_df).

    ADR-0016 Gate 2 (ingestion). Governance/lineage columns (leading `_`) are excluded
    from the scan — they carry ISO timestamps that would otherwise trip the DOB regex
    and reject every row. Envelope structural columns listed in STRUCTURAL_STRING_ALLOWLIST
    (e.g. `simulatedAt`, `emittedAt`, `eventId`) are also excluded to mirror the discipline
    in 02_silver_eventstream.ipynb Gate 2 — cross-notebook parity, even though master-data
    tables don't currently carry those columns. Any hit on a source column aborts the
    notebook after emitting an alert."""
    string_cols = [f.name for f in df.schema.fields
                   if isinstance(f.dataType, T.StringType)
                   and not f.name.startswith('_')
                   and f.name not in STRUCTURAL_STRING_ALLOWLIST]
    if not string_cols:
        return df, df.limit(0)
    hit_expr = None
    for pattern_name, pattern in PHI_PATTERNS.items():
        for col in string_cols:
            regex_lit = pattern.pattern
            match = F.col(col).rlike(regex_lit)
            hit_expr = match if hit_expr is None else (hit_expr | match)
    rejected = df.filter(hit_expr)
    clean = df.filter(~hit_expr)
    n_rej = rejected.count()
    if n_rej > 0:
        print(f'ALERT [{table}] gate 3 PHI regex sweep rejected {n_rej} row(s) at {_now_iso()}')
        raise GateFailure(f'{table}: gate 3 PHI regex sweep failed — {n_rej} row(s) matched PHI patterns; run aborted')
    return clean, rejected

def gate4_residency(df: DataFrame, table: str, residency_source_col):
    if residency_source_col and residency_source_col in df.columns:
        tagged = df.withColumn('_residency_tag', F.coalesce(F.col(residency_source_col), F.lit(default_residency_tag)))
    else:
        tagged = df.withColumn('_residency_tag', F.lit(default_residency_tag))
    bad = tagged.filter(~F.col('_residency_tag').isin(list(ALLOWED_RESIDENCY_TAGS))).count()
    if bad > 0:
        raise GateFailure(f'{table}: gate 4 residency failed — {bad} row(s) with _residency_tag outside {sorted(ALLOWED_RESIDENCY_TAGS)}')
    return tagged

def gate5_data_quality(df: DataFrame, table: str, quality_source_col):
    if quality_source_col and quality_source_col in df.columns:
        tagged = df.withColumn('_data_quality', F.coalesce(F.col(quality_source_col), F.lit('explicit')))
    else:
        tagged = df.withColumn('_data_quality', F.lit('explicit'))
    bad = tagged.filter(~F.col('_data_quality').isin(list(ALLOWED_DATA_QUALITY))).count()
    if bad > 0:
        raise GateFailure(f'{table}: gate 5 data_quality failed — {bad} row(s) with _data_quality outside {sorted(ALLOWED_DATA_QUALITY)}')
    return tagged

def gate6_fk_integrity(df: DataFrame, table: str, fks: dict, silver_cache: dict) -> None:
    if not fks:
        return
    total = df.count() or 1
    for fk_col, parent_table in fks.items():
        parent_pk_col = fk_col  # dim tables use the same column name as the FK by convention
        parent_df = silver_cache.get(parent_table)
        if parent_df is None:
            raise GateFailure(f'{table}: gate 6 FK integrity — parent table {parent_table} not yet in silver cache (load order violation)')
        parent_keys = parent_df.select(F.col(parent_pk_col).alias('__parent_key')).distinct()
        orphans = (df
                     .select(F.col(fk_col).alias('__child_key'))
                     .filter(F.col('__child_key').isNotNull())
                     .join(parent_keys, F.col('__child_key') == F.col('__parent_key'), 'left_anti'))
        n_orphan = orphans.count()
        pct = n_orphan / total
        if pct > orphan_fk_threshold:
            raise GateFailure(f'{table}: gate 6 FK integrity — {n_orphan}/{total} ({pct:.1%}) orphan {fk_col} > {orphan_fk_threshold:.0%} threshold')
        if n_orphan > 0:
            print(f'WARN [{table}] gate 6 FK integrity — {n_orphan}/{total} ({pct:.2%}) orphan {fk_col} (below abort threshold)')

In [ ]:
def promote_table(csv_filename, table_name, pk_cols, fks, residency_source_col, quality_source_col, silver_cache):
    df = spark.table(f'bronze.{table_name}')

    n = gate1_row_count(df, table_name)
    gate2_schema(df, table_name, pk_cols)
    clean, _rejected = gate3_phi_scan(df, table_name)
    clean = gate4_residency(clean, table_name, residency_source_col)
    clean = gate5_data_quality(clean, table_name, quality_source_col)
    gate6_fk_integrity(clean, table_name, fks, silver_cache)

    (clean.write
          .format('delta')
          .mode('overwrite')
          .option('overwriteSchema', 'true')
          .saveAsTable(f'silver.{table_name}'))
    silver_cache[table_name] = clean
    return n, clean.count()

In [ ]:
# Ensure the silver schema exists (schema-enabled lakehouse managed tables).
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Promote all 9 tables in FK-safe order (registry order respects the load graph).
silver_cache = {}
results = []
for csv_filename, table_name, pk_cols, fks, residency_col, quality_col in TABLES:
    n_in, n_out = promote_table(csv_filename, table_name, pk_cols, fks, residency_col, quality_col, silver_cache)
    results.append((table_name, n_in, n_out))

print('Silver promotion summary (run_id=%s)' % run_id)
print('-' * 72)
for t, n_in, n_out in results:
    print(f'{t:<48s} bronze_rows={n_in:<6d} silver_rows={n_out:<6d}')

## PHI regex gate — deterministic self-test

Seed one synthetic bad row into a copy of `dim_hospital` (fake email injected into
`city`) and confirm Gate 3 rejects it. The assertion below fails loudly if the
regex ever regresses. Test is fully in-memory — bronze and silver Delta paths are
untouched.

**Expected output**: an `ALERT` line naming `dim_hospital__phi_test`, followed by
`GateFailure` raised and captured, then `PHI GATE TEST: PASSED`.

In [ ]:
# Deterministic PHI gate self-test.
from pyspark.sql import Row

test_rows = [
    Row(hospital_id='H_TEST_OK',  name='Ok Hospital',   city='Zurich',                   _lineage_ref='test:2026-07-03T00:00:00Z'),
    Row(hospital_id='H_TEST_BAD', name='Bad Hospital',  city='contact@example.com',      _lineage_ref='test:2026-07-03T00:00:00Z'),  # fake email in source col — must be rejected
]
test_df = spark.createDataFrame(test_rows)

gate_raised = False
gate_message = None
try:
    gate3_phi_scan(test_df, 'dim_hospital__phi_test')
except GateFailure as exc:
    gate_raised = True
    gate_message = str(exc)

assert gate_raised, 'PHI gate did NOT raise on synthetic bad row — regression detected'
assert 'dim_hospital__phi_test' in gate_message, 'PHI gate raised but table name missing from error'
assert '1 row(s) matched' in gate_message, f'Expected exactly 1 rejected row; got: {gate_message}'
print('PHI GATE TEST: PASSED')
print(f'  captured: {gate_message}')